<a href="https://colab.research.google.com/github/Agusricc78/MachineLearning/blob/main/TpIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

class DataLoader:
    """Clase para cargar los datos."""
    def load_data(self, filepath, encoding='utf-8'):
        print(f"Cargando datos desde {filepath} con encoding {encoding}...")
        return pd.read_csv(filepath, encoding=encoding)

class DataCleaner:
    """Clase para limpiar los datos."""
    def clean(self, df):
        print("Limpiando datos...")
        df_cleaned = df.dropna(subset=['valor'])
        irrelevant_cols = ['sector_id', 'sector_nombre', 'alcance_id']
        df_cleaned = df_cleaned.drop(columns=irrelevant_cols, errors='ignore') # errors='ignore' por si ya fue borrada
        print(f"Datos limpios. {len(df) - len(df_cleaned)} filas eliminadas.")
        return df_cleaned

class FeatureEngineer:
    """Clase para crear nuevas características."""
    def engineer_features(self, df):
        print("Realizando ingeniería de características...")
        df_engineered = df.copy()
        df_engineered['indice_tiempo'] = pd.to_datetime(df_engineered['indice_tiempo'])
        df_engineered['Año'] = df_engineered['indice_tiempo'].dt.year
        df_engineered['Mes'] = df_engineered['indice_tiempo'].dt.month
        df_engineered['Día'] = df_engineered['indice_tiempo'].dt.day
        df_engineered = df_engineered.drop(columns=['indice_tiempo'])
        return df_engineered

class MLPipeline:
    """Clase para construir el pipeline de preprocesamiento y modelo."""
    def __init__(self, numeric_features, categorical_features):
        self.numeric_features = numeric_features
        self.categorical_features = categorical_features

    def create_pipeline(self):
        print("Creando el pipeline de ML...")
        numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
        categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ])

        model_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
        ])
        return model_pipeline

class ModelTrainer:
    """Clase para entrenar el modelo."""
    def train(self, pipeline, X_train, y_train):
        print("Entrenando el modelo...")
        pipeline.fit(X_train, y_train)
        print("Modelo entrenado exitosamente.")
        return pipeline

class ModelEvaluator:
    """Clase para evaluar el rendimiento del modelo."""
    def evaluate(self, pipeline, X_test, y_test):
        print("Evaluando el modelo...")
        y_pred = pipeline.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        print("\n--- Métricas de Evaluación ---")
        print(f"R² (R-squared): {r2:.4f}")
        print(f"MAE (Error Absoluto Medio): {mae:,.2f}")
        print("-------------------------------")

class Predictor:
    """Clase para realizar nuevas predicciones."""
    def predict_new(self, pipeline, data_dict):
        new_data = pd.DataFrame([data_dict])
        prediction = pipeline.predict(new_data)
        print("\n--- Predicción Práctica ---")
        print(f"Datos de entrada: {data_dict}")
        print(f"Valor predicho: {prediction[0]:,.2f}")
        return prediction

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [19]:
def main():
    """Función principal para orquestar el flujo de ML."""

    # 1. Cargar Datos (El archivo debe estar subido)
    loader = DataLoader()
    df = loader.load_data('azucar.csv', encoding='latin1')

    # 2. Limpiar Datos
    cleaner = DataCleaner()
    df_clean = cleaner.clean(df)

    # 3. Ingeniería de Características
    engineer = FeatureEngineer()
    df_final = engineer.engineer_features(df_clean)

    # 4. Definir características (X) y objetivo (y)
    TARGET = 'valor'
    NUMERIC_FEATURES = ['Año', 'Mes', 'Día']
    CATEGORICAL_FEATURES = [
        'variable_id', 'actividad_producto_nombre', 'indicador',
        'unidad_de_medida', 'fuente', 'frecuencia_nombre',
        'cobertura_nombre', 'alcance_tipo', 'alcance_nombre'
    ]

    X = df_final[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y = df_final[TARGET]

    # 5. Separar en Entrenamiento y Prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Datos separados: {len(X_train)} para entrenar, {len(X_test)} para probar.")

    # 6. Crear Pipeline
    pipeline_builder = MLPipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
    model_pipeline = pipeline_builder.create_pipeline()

    # 7. Entrenar Modelo
    trainer = ModelTrainer()
    trained_pipeline = trainer.train(model_pipeline, X_train, y_train)

    # 8. Evaluar Modelo
    evaluator = ModelEvaluator()
    evaluator.evaluate(trained_pipeline, X_test, y_test)

    # 9. Predicción Práctica (Ejemplo)
    ejemplo_nuevo = {
        'Año': 2024,
        'Mes': 10,
        'Día': 20,
        'variable_id': 22, # ID para "Elaboración"
        'actividad_producto_nombre': 'Azúcar',
        'indicador': 'Elaboración',
        'unidad_de_medida': 'TMVC',
        'fuente': 'Centro Azucarero Argentino',
        'frecuencia_nombre': 'Anual',
        'cobertura_nombre': 'Nacional',
        'alcance_tipo': 'INGENIO',
        'alcance_nombre': 'La Esperanza'
    }

    predictor = Predictor()
    predictor.predict_new(trained_pipeline, ejemplo_nuevo)

# --- ¡EJECUTAR EL PROYECTO! ---
# (Esto corre la función que acabamos de definir)
main()

Cargando datos desde azucar.csv con encoding latin1...
Limpiando datos...
Datos limpios. 240 filas eliminadas.
Realizando ingeniería de características...
Datos separados: 2449 para entrenar, 613 para probar.
Creando el pipeline de ML...
Entrenando el modelo...
Modelo entrenado exitosamente.
Evaluando el modelo...

--- Métricas de Evaluación ---
R² (R-squared): 0.9640
MAE (Error Absoluto Medio): 6,723.50
-------------------------------

--- Predicción Práctica ---
Datos de entrada: {'Año': 2024, 'Mes': 10, 'Día': 20, 'variable_id': 22, 'actividad_producto_nombre': 'Azúcar', 'indicador': 'Elaboración', 'unidad_de_medida': 'TMVC', 'fuente': 'Centro Azucarero Argentino', 'frecuencia_nombre': 'Anual', 'cobertura_nombre': 'Nacional', 'alcance_tipo': 'INGENIO', 'alcance_nombre': 'La Esperanza'}
Valor predicho: 72,693.22


In [ ]:
from google.colab import drive
drive.mount('/content/drive')